In [1]:
import numpy as np
import pandas as pd
import time
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
df = pd.read_csv("garments_worker_productivity.csv")

df.columns = df.columns.str.strip()

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [3]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nColumns:")
print(df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

In [4]:
df = df.copy()

if "date" in df.columns:
    df = df.drop(columns=["date"])

df = df.dropna(axis=1, how="all")

df["MeetsTarget"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)

print(df.head())

print("\nTarget Distribution:")
print(df["MeetsTarget"].value_counts())

    quarter  department       day  team  targeted_productivity    smv     wip  \
0  Quarter1      sweing  Thursday     8                   0.80  26.16  1108.0   
1  Quarter1  finishing   Thursday     1                   0.75   3.94     NaN   
2  Quarter1      sweing  Thursday    11                   0.80  11.41   968.0   
3  Quarter1      sweing  Thursday    12                   0.80  11.41   968.0   
4  Quarter1      sweing  Thursday     6                   0.80  25.90  1170.0   

   over_time  incentive  idle_time  idle_men  no_of_style_change  \
0       7080         98        0.0         0                   0   
1        960          0        0.0         0                   0   
2       3660         50        0.0         0                   0   
3       3660         50        0.0         0                   0   
4       1920         50        0.0         0                   0   

   no_of_workers  actual_productivity  MeetsTarget  
0           59.0             0.940725            1 

In [5]:
# Regression target
X_reg = df.drop(columns=["actual_productivity", "MeetsTarget"])
y_reg = df["actual_productivity"]

# Classification target
# actual_productivity is NOT used as an input
X_clf = df.drop(columns=["actual_productivity", "MeetsTarget"])
y_clf = df["MeetsTarget"]

print("Regression Features:")
print(X_reg.columns.tolist())

print("\nClassification Features:")
print(X_clf.columns.tolist())

Regression Features:
['quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Classification Features:
['quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


In [6]:
# One fixed train-test split for ALL implementations

indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(train_idx))
print("Testing samples :", len(test_idx))

Training samples: 957
Testing samples : 240


In [7]:
numeric_cols = X_reg.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_cols = X_reg.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric Columns:")
print(numeric_cols)

print("\nCategorical Columns:")
print(categorical_cols)

Numeric Columns:
['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Categorical Columns:
['quarter', 'department', 'day']


In [8]:
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder)
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [9]:
X_train_reg = X_reg.iloc[train_idx].copy()
X_test_reg = X_reg.iloc[test_idx].copy()

y_train_reg = y_reg.iloc[train_idx].copy()
y_test_reg = y_reg.iloc[test_idx].copy()

X_train_clf = X_clf.iloc[train_idx].copy()
X_test_clf = X_clf.iloc[test_idx].copy()

y_train_clf = y_clf.iloc[train_idx].copy()
y_test_clf = y_clf.iloc[test_idx].copy()

print("Train shape:", X_train_reg.shape)
print("Test shape :", X_test_reg.shape)

Train shape: (957, 13)
Test shape : (240, 13)


In [10]:
# Scikit-learn preprocessing

X_train_processed = preprocessor.fit_transform(X_train_reg)
X_test_processed = preprocessor.transform(X_test_reg)

print("Processed Train Shape:", X_train_processed.shape)
print("Processed Test Shape :", X_test_processed.shape)

feature_names = preprocessor.get_feature_names_out()
print("\nNumber of Features:", len(feature_names))

Processed Train Shape: (957, 24)
Processed Test Shape : (240, 24)

Number of Features: 24


In [11]:
# Scikit-learn Linear Regression

start = time.perf_counter()

linear_model = LinearRegression()
linear_model.fit(X_train_processed, y_train_reg)

sk_linear_train_time = time.perf_counter() - start

start = time.perf_counter()

y_pred_reg = linear_model.predict(X_test_processed)

sk_linear_prediction_time = time.perf_counter() - start

sk_mae = mean_absolute_error(y_test_reg, y_pred_reg)
sk_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
sk_r2 = r2_score(y_test_reg, y_pred_reg)

print("Scikit-learn Linear Regression")
print("MAE :", sk_mae)
print("RMSE:", sk_rmse)
print("R2  :", sk_r2)
print("Training Time :", sk_linear_train_time)
print("Prediction Time:", sk_linear_prediction_time)

Scikit-learn Linear Regression
MAE : 0.10845261337977505
RMSE: 0.14861757809593648
R2  : 0.16816825663056223
Training Time : 0.013843199936673045
Prediction Time: 0.0007522000232711434


In [12]:
# Scikit-learn Logistic Regression

start = time.perf_counter()

logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train_clf
)

sk_logistic_train_time = time.perf_counter() - start

start = time.perf_counter()

y_pred_clf = logistic_model.predict(X_test_processed)

sk_logistic_prediction_time = time.perf_counter() - start

sk_accuracy = accuracy_score(y_test_clf, y_pred_clf)

sk_precision = precision_score(
    y_test_clf, y_pred_clf, zero_division=0
)

sk_recall = recall_score(
    y_test_clf, y_pred_clf, zero_division=0
)

sk_f1 = f1_score(
    y_test_clf, y_pred_clf, zero_division=0
)

print("Scikit-learn Logistic Regression")
print("Accuracy :", sk_accuracy)
print("Precision:", sk_precision)
print("Recall   :", sk_recall)
print("F1 Score :", sk_f1)
print("Training Time :", sk_logistic_train_time)
print("Prediction Time:", sk_logistic_prediction_time)

Scikit-learn Logistic Regression
Accuracy : 0.775
Precision: 0.7808219178082192
Recall   : 0.9661016949152542
F1 Score : 0.8636363636363636
Training Time : 0.12029979994986206
Prediction Time: 0.00035360001493245363


In [13]:
# Manual preprocessing using ONLY Pandas and NumPy

X_manual = X_reg.copy()

# Numeric missing values
X_num = X_manual[numeric_cols].copy()

for col in numeric_cols:
    X_num[col] = X_num[col].fillna(X_num[col].median())

# Categorical missing values
X_cat = X_manual[categorical_cols].copy()

for col in categorical_cols:
    X_cat[col] = X_cat[col].fillna(X_cat[col].mode()[0])

# Manual one-hot encoding
X_cat_encoded = pd.get_dummies(
    X_cat,
    columns=categorical_cols,
    dtype=float
)

# Combine
X_manual_processed = pd.concat(
    [X_num, X_cat_encoded],
    axis=1
)

X_manual_processed = X_manual_processed.astype(float)

print("Manual Processed Shape:", X_manual_processed.shape)

Manual Processed Shape: (1197, 24)


In [14]:
# Manual scaling
# Statistics are calculated using TRAINING DATA only

X_manual_train = X_manual_processed.iloc[train_idx].copy()
X_manual_test = X_manual_processed.iloc[test_idx].copy()

manual_means = X_manual_train.mean()
manual_stds = X_manual_train.std(ddof=0)

manual_stds = manual_stds.replace(0, 1)

X_manual_train_scaled = (
    X_manual_train - manual_means
) / manual_stds

X_manual_test_scaled = (
    X_manual_test - manual_means
) / manual_stds

X_manual_train_scaled = X_manual_train_scaled.to_numpy()
X_manual_test_scaled = X_manual_test_scaled.to_numpy()

print("Train:", X_manual_train_scaled.shape)
print("Test :", X_manual_test_scaled.shape)

Train: (957, 24)
Test : (240, 24)


In [15]:
# Add intercept column for manual models

X_train_manual = np.column_stack([
    np.ones(X_manual_train_scaled.shape[0]),
    X_manual_train_scaled
])

X_test_manual = np.column_stack([
    np.ones(X_manual_test_scaled.shape[0]),
    X_manual_test_scaled
])

y_manual_train_reg = y_reg.iloc[train_idx].to_numpy(dtype=float)
y_manual_test_reg = y_reg.iloc[test_idx].to_numpy(dtype=float)

y_manual_train_clf = y_clf.iloc[train_idx].to_numpy(dtype=float)
y_manual_test_clf = y_clf.iloc[test_idx].to_numpy(dtype=float)

print("Manual Train Matrix:", X_train_manual.shape)
print("Manual Test Matrix :", X_test_manual.shape)

Manual Train Matrix: (957, 25)
Manual Test Matrix : (240, 25)


In [16]:
# Manual Linear Regression
# Closed-form solution:
# beta = (X^T X)^-1 X^T y

start = time.perf_counter()

XTX = X_train_manual.T @ X_train_manual
XTy = X_train_manual.T @ y_manual_train_reg

beta = np.linalg.pinv(XTX) @ XTy

manual_linear_train_time = time.perf_counter() - start

start = time.perf_counter()

y_manual_pred_reg = X_test_manual @ beta

manual_linear_prediction_time = time.perf_counter() - start

print("Manual Linear Regression completed")
print("Training Time :", manual_linear_train_time)
print("Prediction Time:", manual_linear_prediction_time)

Manual Linear Regression completed
Training Time : 0.09346230002120137
Prediction Time: 4.339998122304678e-05


In [17]:
# Manual regression metrics

def manual_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def manual_rmse(y_true, y_pred):
    return np.sqrt(
        np.mean((y_true - y_pred) ** 2)
    )


def manual_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum(
        (y_true - np.mean(y_true)) ** 2
    )
    return 1 - (ss_res / ss_tot)


manual_mae_value = manual_mae(
    y_manual_test_reg,
    y_manual_pred_reg
)

manual_rmse_value = manual_rmse(
    y_manual_test_reg,
    y_manual_pred_reg
)

manual_r2_value = manual_r2(
    y_manual_test_reg,
    y_manual_pred_reg
)

print("Manual Linear Regression")
print("MAE :", manual_mae_value)
print("RMSE:", manual_rmse_value)
print("R2  :", manual_r2_value)

Manual Linear Regression
MAE : 0.10845261337977516
RMSE: 0.14861757809593654
R2  : 0.16816825663056179


In [18]:
# Sigmoid function

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


# Manual Logistic Regression

def train_logistic_regression(
    X,
    y,
    learning_rate=0.05,
    epochs=10000,
    tolerance=1e-7
):
    n_samples, n_features = X.shape

    weights = np.zeros(n_features)
    previous_loss = float("inf")

    for epoch in range(epochs):

        z = X @ weights
        probabilities = sigmoid(z)

        gradient = (
            X.T @ (probabilities - y)
        ) / n_samples

        weights -= learning_rate * gradient

        p = np.clip(
            probabilities,
            1e-15,
            1 - 1e-15
        )

        loss = -np.mean(
            y * np.log(p)
            + (1 - y) * np.log(1 - p)
        )

        if abs(previous_loss - loss) < tolerance:
            break

        previous_loss = loss

    return weights, epoch + 1, loss

In [19]:
# Train manual Logistic Regression

start = time.perf_counter()

manual_logistic_weights, epochs_used, final_loss = (
    train_logistic_regression(
        X_train_manual,
        y_manual_train_clf,
        learning_rate=0.05,
        epochs=10000,
        tolerance=1e-7
    )
)

manual_logistic_train_time = time.perf_counter() - start

print("Manual Logistic Regression")
print("Epochs:", epochs_used)
print("Final Loss:", final_loss)
print("Training Time:", manual_logistic_train_time)

Manual Logistic Regression
Epochs: 6309
Final Loss: 0.4881214228252558
Training Time: 0.36312250001356006


In [20]:
# Manual Logistic Prediction

start = time.perf_counter()

manual_probabilities = sigmoid(
    X_test_manual @ manual_logistic_weights
)

manual_pred_clf = (
    manual_probabilities >= 0.5
).astype(int)

manual_logistic_prediction_time = (
    time.perf_counter() - start
)

print("Prediction Time:", manual_logistic_prediction_time)

Prediction Time: 0.00018209998961538076


In [21]:
# Manual classification metrics

def manual_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


def manual_precision(y_true, y_pred):
    tp = np.sum(
        (y_true == 1) & (y_pred == 1)
    )
    fp = np.sum(
        (y_true == 0) & (y_pred == 1)
    )

    if tp + fp == 0:
        return 0

    return tp / (tp + fp)


def manual_recall(y_true, y_pred):
    tp = np.sum(
        (y_true == 1) & (y_pred == 1)
    )
    fn = np.sum(
        (y_true == 1) & (y_pred == 0)
    )

    if tp + fn == 0:
        return 0

    return tp / (tp + fn)


def manual_f1(y_true, y_pred):
    p = manual_precision(y_true, y_pred)
    r = manual_recall(y_true, y_pred)

    if p + r == 0:
        return 0

    return 2 * p * r / (p + r)


manual_accuracy_value = manual_accuracy(
    y_manual_test_clf,
    manual_pred_clf
)

manual_precision_value = manual_precision(
    y_manual_test_clf,
    manual_pred_clf
)

manual_recall_value = manual_recall(
    y_manual_test_clf,
    manual_pred_clf
)

manual_f1_value = manual_f1(
    y_manual_test_clf,
    manual_pred_clf
)

print("Accuracy :", manual_accuracy_value)
print("Precision:", manual_precision_value)
print("Recall   :", manual_recall_value)
print("F1 Score :", manual_f1_value)

Accuracy : 0.775
Precision: 0.783410138248848
Recall   : 0.96045197740113
F1 Score : 0.8629441624365483


In [22]:
# Optimized Manual Logistic Regression
# Vectorized + L2 regularization

def train_optimized_logistic(
    X,
    y,
    learning_rate=0.03,
    epochs=20000,
    regularization=0.001,
    tolerance=1e-8
):
    n_samples, n_features = X.shape

    weights = np.zeros(n_features)
    previous_loss = float("inf")

    for epoch in range(epochs):

        z = X @ weights
        probabilities = sigmoid(z)

        gradient = (
            X.T @ (probabilities - y)
        ) / n_samples

        gradient[1:] += (
            regularization * weights[1:]
        )

        weights -= learning_rate * gradient

        p = np.clip(
            probabilities,
            1e-15,
            1 - 1e-15
        )

        loss = -np.mean(
            y * np.log(p)
            + (1 - y) * np.log(1 - p)
        )

        loss += (
            regularization / 2
        ) * np.sum(weights[1:] ** 2)

        if abs(previous_loss - loss) < tolerance:
            break

        previous_loss = loss

    return weights, epoch + 1, loss


start = time.perf_counter()

optimized_weights, optimized_epochs, optimized_loss = (
    train_optimized_logistic(
        X_train_manual,
        y_manual_train_clf,
        learning_rate=0.03,
        epochs=20000,
        regularization=0.001,
        tolerance=1e-8
    )
)

optimized_train_time = time.perf_counter() - start

print("Optimized Manual Model")
print("Epochs:", optimized_epochs)
print("Loss:", optimized_loss)
print("Training Time:", optimized_train_time)

Optimized Manual Model
Epochs: 13680
Loss: 0.49023058567010747
Training Time: 0.6551007999805734


In [23]:
# Optimized prediction and evaluation

start = time.perf_counter()

optimized_probabilities = sigmoid(
    X_test_manual @ optimized_weights
)

optimized_predictions = (
    optimized_probabilities >= 0.5
).astype(int)

optimized_prediction_time = (
    time.perf_counter() - start
)

optimized_accuracy = manual_accuracy(
    y_manual_test_clf,
    optimized_predictions
)

optimized_precision = manual_precision(
    y_manual_test_clf,
    optimized_predictions
)

optimized_recall = manual_recall(
    y_manual_test_clf,
    optimized_predictions
)

optimized_f1 = manual_f1(
    y_manual_test_clf,
    optimized_predictions
)

print("Optimized Manual Logistic Regression")
print("Accuracy :", optimized_accuracy)
print("Precision:", optimized_precision)
print("Recall   :", optimized_recall)
print("F1 Score :", optimized_f1)
print("Prediction Time:", optimized_prediction_time)

Optimized Manual Logistic Regression
Accuracy : 0.775
Precision: 0.783410138248848
Recall   : 0.96045197740113
F1 Score : 0.8629441624365483
Prediction Time: 0.00019189994782209396


In [24]:
# Regression Comparison

regression_comparison = pd.DataFrame({
    "Implementation": [
        "Scikit-learn",
        "From Scratch"
    ],
    "MAE": [
        sk_mae,
        manual_mae_value
    ],
    "RMSE": [
        sk_rmse,
        manual_rmse_value
    ],
    "R2": [
        sk_r2,
        manual_r2_value
    ],
    "Training Time (sec)": [
        sk_linear_train_time,
        manual_linear_train_time
    ],
    "Prediction Time (sec)": [
        sk_linear_prediction_time,
        manual_linear_prediction_time
    ]
})

display(regression_comparison)

,Implementation,MAE,RMSE,R2,Training Time (sec),Prediction Time (sec)
0,Scikit-learn,0.108453,0.148618,0.168168,0.013843,0.000752
1,From Scratch,0.108453,0.148618,0.168168,0.093462,0.000043


In [25]:
# Classification Comparison

classification_comparison = pd.DataFrame({
    "Implementation": [
        "Scikit-learn",
        "From Scratch",
        "Optimized From Scratch"
    ],
    "Accuracy": [
        sk_accuracy,
        manual_accuracy_value,
        optimized_accuracy
    ],
    "Precision": [
        sk_precision,
        manual_precision_value,
        optimized_precision
    ],
    "Recall": [
        sk_recall,
        manual_recall_value,
        optimized_recall
    ],
    "F1 Score": [
        sk_f1,
        manual_f1_value,
        optimized_f1
    ],
    "Training Time (sec)": [
        sk_logistic_train_time,
        manual_logistic_train_time,
        optimized_train_time
    ],
    "Prediction Time (sec)": [
        sk_logistic_prediction_time,
        manual_logistic_prediction_time,
        optimized_prediction_time
    ]
})

display(classification_comparison)

,Implementation,Accuracy,Precision,Recall,F1 Score,Training Time (sec),Prediction Time (sec)
0,Scikit-learn,0.775,0.780822,0.966102,0.863636,0.120300,0.000354
1,From Scratch,0.775,0.783410,0.960452,0.862944,0.363123,0.000182
2,Optimized From Scratch,0.775,0.783410,0.960452,0.862944,0.655101,0.000192


In [26]:
# Save comparison results

regression_comparison.to_csv(
    "regression_comparison.csv",
    index=False
)

classification_comparison.to_csv(
    "classification_comparison.csv",
    index=False
)

results = pd.DataFrame({
    "Metric": [
        "Sklearn Regression MAE",
        "Sklearn Regression RMSE",
        "Sklearn Regression R2",
        "Manual Regression MAE",
        "Manual Regression RMSE",
        "Manual Regression R2",
        "Sklearn Classification Accuracy",
        "Sklearn Classification Precision",
        "Sklearn Classification Recall",
        "Sklearn Classification F1",
        "Manual Classification Accuracy",
        "Manual Classification Precision",
        "Manual Classification Recall",
        "Manual Classification F1",
        "Optimized Manual Accuracy",
        "Optimized Manual Precision",
        "Optimized Manual Recall",
        "Optimized Manual F1"
    ],
    "Value": [
        sk_mae,
        sk_rmse,
        sk_r2,
        manual_mae_value,
        manual_rmse_value,
        manual_r2_value,
        sk_accuracy,
        sk_precision,
        sk_recall,
        sk_f1,
        manual_accuracy_value,
        manual_precision_value,
        manual_recall_value,
        manual_f1_value,
        optimized_accuracy,
        optimized_precision,
        optimized_recall,
        optimized_f1
    ]
})

results.to_csv(
    "final_results.csv",
    index=False
)

display(results)

print("All result files saved successfully.")

,Metric,Value
0,Sklearn Regression MAE,0.108453
1,Sklearn Regression RMSE,0.148618
2,Sklearn Regression R2,0.168168
3,Manual Regression MAE,0.108453
4,Manual Regression RMSE,0.148618
5,Manual Regression R2,0.168168
6,Sklearn Classification Accuracy,0.775000
7,Sklearn Classification Precision,0.780822
8,Sklearn Classification Recall,0.966102
9,Sklearn Classification F1,0.863636


All result files saved successfully.


In [27]:
# Final verification

print("=" * 60)
print("DS605 LAB ASSIGNMENT 5 - FINAL VERIFICATION")
print("=" * 60)

print("\nDataset:")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nFixed Train-Test Split:")
print("Training:", len(train_idx))
print("Testing :", len(test_idx))

print("\nRegression:")
print("Target = actual_productivity")

print("\nClassification:")
print("Target = MeetsTarget")
print("MeetsTarget = 1 when actual_productivity >= targeted_productivity")
print("actual_productivity used as classification input: NO")

print("\n--- Regression ---")
print("Scikit-learn R2:", sk_r2)
print("Manual R2      :", manual_r2_value)

print("\n--- Classification ---")
print("Scikit-learn F1       :", sk_f1)
print("Manual F1             :", manual_f1_value)
print("Optimized Manual F1   :", optimized_f1)

print("\nFiles created:")
print("1. regression_comparison.csv")
print("2. classification_comparison.csv")
print("3. final_results.csv")

print("\nAssignment workflow completed successfully.")

DS605 LAB ASSIGNMENT 5 - FINAL VERIFICATION

Dataset:
Rows: 1197
Columns: 15

Fixed Train-Test Split:
Training: 957
Testing : 240

Regression:
Target = actual_productivity

Classification:
Target = MeetsTarget
MeetsTarget = 1 when actual_productivity >= targeted_productivity
actual_productivity used as classification input: NO

--- Regression ---
Scikit-learn R2: 0.16816825663056223
Manual R2      : 0.16816825663056179

--- Classification ---
Scikit-learn F1       : 0.8636363636363636
Manual F1             : 0.8629441624365483
Optimized Manual F1   : 0.8629441624365483

Files created:
1. regression_comparison.csv
2. classification_comparison.csv
3. final_results.csv

Assignment workflow completed successfully.
